In [2]:
import myQMLlib as myQML
import numpy as np
import matplotlib.pyplot as plt
import time
import json
import os

#Code to automatically reload the myQMLfunctions module when it is edited
%load_ext autoreload
%autoreload 2

In [2]:
####### Settings #########
# Put ONLY the values you want to simulate right now in this list.
# If you run [10, 30] today, and change this to [50, 100] tomorrow, the code will merge them!
N_train_list = [2,3,4,5,6,7,8,9,10,20,30,40,50,64,75,100,150,200,500,750,1000]  
N_test = 250
num_realizations = 500
num_shots = 1000
reg_lambda = 1/np.sqrt(num_shots)  # regularization parameter for kernel regression
filename = "1000_shots_ridge.json"

# --- LOAD PREVIOUS DATA IF IT EXISTS ---
if os.path.exists(filename):
    print(f"Found existing file '{filename}'. Loading previous data...")
    with open(filename, 'r') as f:
        saved_data = json.load(f)
else:
    print("No previous data found. Starting fresh.")
    saved_data = {'QELM': {}, 'Kernel_LE': {}, 'Kernel_SWAP': {}}

# Filter N_train_list to ONLY calculate values we haven't done yet
# (JSON saves dictionary keys as strings, so we check str(N))
N_train_to_run = [N for N in N_train_list if str(N) not in saved_data['QELM']]

if not N_train_to_run:
    print("All N_train values in the list have already been simulated. Exiting.")
    exit()

print(f"Simulating for N_train = {N_train_to_run}")

N_max = max(N_train_to_run)

# QELM reservoir physical setup
d_in = 2
d_res = 64
d_out = d_in * d_res
povm = myQML.generate_computational_povm(d_res) 
num_povm_elements = len(povm)

# Temporary dictionary for the CURRENT run
mse_runs = {
    'QELM': {N: [] for N in N_train_to_run},
    'Kernel_LE': {N: [] for N in N_train_to_run},
    'Kernel_SWAP': {N: [] for N in N_train_to_run}
}

print(f"-------- (Shots = {num_shots}, Realizations = {num_realizations}) --------")
print("="*70)

total_time = time.time()

# INVERTED LOOP
for r in range(num_realizations):
    if (r + 1) % 10 == 0:
        print(f"Processing Realization {r + 1}/{num_realizations}...")
    
    #random Pauli observable
    observable = myQML.random_pauli()

    #generate dataset and compute expectation values
    ds = myQML.QuantumDatasetGenerator(N_max, N_test, observable)
    ds.generate_density_matrices_vec()
    ds.compute_expectation_values_vec()

    rho_train_full, y_train_full = ds.get_training_dataset()
    rho_test, y_test = ds.get_test_dataset()

    kernel_le_full = myQML.QuantumKernelRegression(reg_lambda, num_shots)
    kernel_le_full.fit_vec(rho_train_full, y_train_full, "le")
    K_matrix_le_full = kernel_le_full.kernel_matrix 

    kernel_swap_full = myQML.QuantumKernelRegression(reg_lambda, num_shots)
    kernel_swap_full.fit_vec(rho_train_full, y_train_full, "swap")
    K_matrix_swap_full = kernel_swap_full.kernel_matrix

    #putting V here, we are assuming that for each realization the input state of
    #the reservoir is the same, and that the only thing that changes is the training/test states
    V = myQML.random_isometry(d_in, d_in * d_res)
    qelm = myQML.QuantumExtremeLearningMachine(
        isometry=V, povm=povm, bipartite_dims=(d_in, d_res), 
        keep_subsystem=1, num_shots=num_shots
    )

    for N_train in sorted(N_train_to_run, reverse=True):
        rho_train_sub = rho_train_full[:N_train]
        y_train_sub = y_train_full[:N_train]

        # QELM
        qelm.fit_vec(rho_train_sub, y_train_sub)
        mse_runs['QELM'][N_train].append(float(np.mean((qelm.predict_vec(rho_test) - y_test)**2)))
        
        # Kernel LE
        K_le_sub = K_matrix_le_full[:N_train, :N_train]
        K_le_inv_sub = np.linalg.pinv(K_le_sub, rcond=kernel_le_full.r_cond)
        alpha_le_sub = K_le_inv_sub @ y_train_sub

        kernel_le_full.train_density_matrices = rho_train_sub
        kernel_le_full.train_labels = y_train_sub
        kernel_le_full.kernel_matrix = K_le_sub
        kernel_le_full.K_inv = K_le_inv_sub
        kernel_le_full.alpha = alpha_le_sub
        
        mse_runs['Kernel_LE'][N_train].append(float(np.mean((kernel_le_full.predict_vec(rho_test) - y_test)**2)))

        # Kernel SWAP
        K_swap_sub = K_matrix_swap_full[:N_train, :N_train]
        K_swap_inv_sub = np.linalg.pinv(K_swap_sub, rcond=kernel_swap_full.r_cond)
        alpha_swap_sub = K_swap_inv_sub @ y_train_sub

        kernel_swap_full.train_density_matrices = rho_train_sub
        kernel_swap_full.train_labels = y_train_sub
        kernel_swap_full.kernel_matrix = K_swap_sub
        kernel_swap_full.K_inv = K_swap_inv_sub
        kernel_swap_full.alpha = alpha_swap_sub
        
        mse_runs['Kernel_SWAP'][N_train].append(float(np.mean((kernel_swap_full.predict_vec(rho_test) - y_test)**2)))

# --- MERGE AND SAVE DATA ---
for model in mse_runs.keys():
    for N_train, mse_list in mse_runs[model].items():
        # JSON requires dictionary keys to be strings
        saved_data[model][str(N_train)] = mse_list

with open(filename, 'w') as f:
    json.dump(saved_data, f, indent=4)

print("="*70)
print(f"Experiment finished and data appended to '{filename}' in {(time.time() - total_time)/60:.1f} minutes.")

No previous data found. Starting fresh.
Simulating for N_train = [2, 3, 4, 5, 6, 7, 8, 9, 10, 20, 30, 40, 50, 64, 75, 100, 150, 200, 500, 750, 1000]
-------- (Shots = 1000, Realizations = 500) --------
Processing Realization 10/500...
Processing Realization 20/500...
Processing Realization 30/500...
Processing Realization 40/500...
Processing Realization 50/500...
Processing Realization 60/500...
Processing Realization 70/500...
Processing Realization 80/500...
Processing Realization 90/500...
Processing Realization 100/500...
Processing Realization 110/500...
Processing Realization 120/500...
Processing Realization 130/500...
Processing Realization 140/500...
Processing Realization 150/500...
Processing Realization 160/500...
Processing Realization 170/500...
Processing Realization 180/500...
Processing Realization 190/500...
Processing Realization 200/500...
Processing Realization 210/500...
Processing Realization 220/500...
Processing Realization 230/500...
Processing Realization 24

In [3]:
####### Settings #########
# Put ONLY the values you want to simulate right now in this list.
# If you run [10, 30] today, and change this to [50, 100] tomorrow, the code will merge them!
N_train_list = [2,3,4,5,6,7,8,9,10,20,30,40,50,64,75,100,150,200,500,750,1000]  
N_test = 250
num_realizations = 500
num_shots = 1000
reg_lambda = 1/np.sqrt(num_shots)  # regularization parameter for kernel regression
filename = "100_shots_ridge.json"

# --- LOAD PREVIOUS DATA IF IT EXISTS ---
if os.path.exists(filename):
    print(f"Found existing file '{filename}'. Loading previous data...")
    with open(filename, 'r') as f:
        saved_data = json.load(f)
else:
    print("No previous data found. Starting fresh.")
    saved_data = {'QELM': {}, 'Kernel_LE': {}, 'Kernel_SWAP': {}}

# Filter N_train_list to ONLY calculate values we haven't done yet
# (JSON saves dictionary keys as strings, so we check str(N))
N_train_to_run = [N for N in N_train_list if str(N) not in saved_data['QELM']]

if not N_train_to_run:
    print("All N_train values in the list have already been simulated. Exiting.")
    exit()

print(f"Simulating for N_train = {N_train_to_run}")

N_max = max(N_train_to_run)

# QELM reservoir physical setup
d_in = 2
d_res = 64
d_out = d_in * d_res
povm = myQML.generate_computational_povm(d_res) 
num_povm_elements = len(povm)

# Temporary dictionary for the CURRENT run
mse_runs = {
    'QELM': {N: [] for N in N_train_to_run},
    'Kernel_LE': {N: [] for N in N_train_to_run},
    'Kernel_SWAP': {N: [] for N in N_train_to_run}
}

print(f"-------- (Shots = {num_shots}, Realizations = {num_realizations}) --------")
print("="*70)

total_time = time.time()

# INVERTED LOOP
for r in range(num_realizations):
    if (r + 1) % 10 == 0:
        print(f"Processing Realization {r + 1}/{num_realizations}...")
    
    #random Pauli observable
    observable = myQML.random_pauli()

    #generate dataset and compute expectation values
    ds = myQML.QuantumDatasetGenerator(N_max, N_test, observable)
    ds.generate_density_matrices_vec()
    ds.compute_expectation_values_vec()

    rho_train_full, y_train_full = ds.get_training_dataset()
    rho_test, y_test = ds.get_test_dataset()

    kernel_le_full = myQML.QuantumKernelRegression(reg_lambda, num_shots)
    kernel_le_full.fit_vec(rho_train_full, y_train_full, "le")
    K_matrix_le_full = kernel_le_full.kernel_matrix 

    kernel_swap_full = myQML.QuantumKernelRegression(reg_lambda, num_shots)
    kernel_swap_full.fit_vec(rho_train_full, y_train_full, "swap")
    K_matrix_swap_full = kernel_swap_full.kernel_matrix

    #putting V here, we are assuming that for each realization the input state of
    #the reservoir is the same, and that the only thing that changes is the training/test states
    V = myQML.random_isometry(d_in, d_in * d_res)
    qelm = myQML.QuantumExtremeLearningMachine(
        isometry=V, povm=povm, bipartite_dims=(d_in, d_res), 
        keep_subsystem=1, num_shots=num_shots
    )

    for N_train in sorted(N_train_to_run, reverse=True):
        rho_train_sub = rho_train_full[:N_train]
        y_train_sub = y_train_full[:N_train]

        # QELM
        qelm.fit_vec(rho_train_sub, y_train_sub)
        mse_runs['QELM'][N_train].append(float(np.mean((qelm.predict_vec(rho_test) - y_test)**2)))
        
        # Kernel LE
        K_le_sub = K_matrix_le_full[:N_train, :N_train]
        K_le_inv_sub = np.linalg.pinv(K_le_sub, rcond=kernel_le_full.r_cond)
        alpha_le_sub = K_le_inv_sub @ y_train_sub

        kernel_le_full.train_density_matrices = rho_train_sub
        kernel_le_full.train_labels = y_train_sub
        kernel_le_full.kernel_matrix = K_le_sub
        kernel_le_full.K_inv = K_le_inv_sub
        kernel_le_full.alpha = alpha_le_sub
        
        mse_runs['Kernel_LE'][N_train].append(float(np.mean((kernel_le_full.predict_vec(rho_test) - y_test)**2)))

        # Kernel SWAP
        K_swap_sub = K_matrix_swap_full[:N_train, :N_train]
        K_swap_inv_sub = np.linalg.pinv(K_swap_sub, rcond=kernel_swap_full.r_cond)
        alpha_swap_sub = K_swap_inv_sub @ y_train_sub

        kernel_swap_full.train_density_matrices = rho_train_sub
        kernel_swap_full.train_labels = y_train_sub
        kernel_swap_full.kernel_matrix = K_swap_sub
        kernel_swap_full.K_inv = K_swap_inv_sub
        kernel_swap_full.alpha = alpha_swap_sub
        
        mse_runs['Kernel_SWAP'][N_train].append(float(np.mean((kernel_swap_full.predict_vec(rho_test) - y_test)**2)))

# --- MERGE AND SAVE DATA ---
for model in mse_runs.keys():
    for N_train, mse_list in mse_runs[model].items():
        # JSON requires dictionary keys to be strings
        saved_data[model][str(N_train)] = mse_list

with open(filename, 'w') as f:
    json.dump(saved_data, f, indent=4)

print("="*70)
print(f"Experiment finished and data appended to '{filename}' in {(time.time() - total_time)/60:.1f} minutes.")

No previous data found. Starting fresh.
Simulating for N_train = [2, 3, 4, 5, 6, 7, 8, 9, 10, 20, 30, 40, 50, 64, 75, 100, 150, 200, 500, 750, 1000]
-------- (Shots = 1000, Realizations = 500) --------
Processing Realization 10/500...


KeyboardInterrupt: 